In [0]:
!pip install openpyxl
import pandas as pd

In [0]:
import pyspark.sql.functions as f
import pyspark.sql.types as t
from pyspark.sql.window import Window
from pyspark.sql import functions as F

import re
import os
import pandas as pd
import numpy as np

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
def interpatient(df):   # it creates the block of events during exchange
    df = df[~df['text'].str.contains('AdjustSeq|ServiceSeq', case = False)] # they don't correspond to an examination
    # Drop the rows that are consecutively repeated by checking datetime and sourceID columns
    mask = (df['datetime'].shift(1) == df['datetime']) & (df['sourceID'].shift(1) == df['sourceID'])
    df = df[~mask]
    filtered_df = pd.DataFrame(columns=['datetime','sourceID', 'text', 'timediff','BodyPart', 'PatientID'])
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Initialize variables
    start_block = False
    start_idx = 0
    prev_patient_id = None
    patients = 0    # Count number of patients since we have to reject patient 1 every day
    date0 = df['datetime'].iloc[0].date()
    first_pat = True
    first_ptab = None
    count_print = 0

    for index, row in df.iterrows():
        code = row['sourceID']
        text = row['text']
        date = row['datetime'].date()
        end_idx = None
        indices_100 = df.index[df['sourceID'] == 'MRI_MSR_100'].tolist()

        if date0 != date:   # Restart variables if new day
            patients = 0
            date0 = date
            start_block = False

        if code == "MRI_FRR_257" and first_pat: # take the ptab from the first examination
            first_ptab = row['text'].split()[-1]

        # Start block if sourceID is MSR_104 and the block has not started yet
        if code == 'MRI_MSR_104' and not start_block:
            start_idx = index
            start_block = True
            first_pat = False # First patient measurement has finished, ptab won't be updated

        # End block if sourceID is MSR_100 and the block has started
        if code == 'MRI_MSR_100' and start_block:
            end_idx = index

        # Get patient and body part info from sourceID EXU_95 if the block has started 
        if code == 'MRI_EXU_95' and start_block:
            # Check if the customer ID is the same as the previous block
            patient_id = re.search("Anonymised Patient ID < (.*) ><", text).group(1)
            if patient_id == prev_patient_id: # If patient doesn't change, reset variables and skip the block
                start_block = False
                continue
            else:       # If patient is different, append the block to filtered_df
                prev_patient_id = re.search("Anonymised Patient ID < (.*) >", text).group(1)
                body_part = re.search("with body part < (.*) > <", text).group(1)
                df_copy  = df.copy()

                # In case MRI_EXU_95 is detected earlier than MRI_MSR_100. Take from the list of MRI_MSR_100 the nearest after the start flag.
                if not end_idx:
                    end_idx = min([x for x in indices_100 if x > start_idx])
                
                # Select interval of interest and calculate time difference
                block = df_copy.loc[start_idx:end_idx]
                if block.empty:
                    block = df_copy.loc[start_idx:index]
                else:
                    time_diff_seconds = (block['datetime'] - block['datetime'].iloc[0]).dt.total_seconds() # accumulated
                    # block['timediff'] = time_diff_seconds
                    block = block.assign(timediff=time_diff_seconds)
                #block['timediff'] = time_diff_seconds
                block = block.assign(timediff=time_diff_seconds)

                # Append the block to the filtered dataframe if it is not the first patient
                patients += 1
                if patients > 1:
                    filtered_df = pd.concat([filtered_df, block], ignore_index=True)

                exu95_row = pd.DataFrame({'sourceID': [''], 'datetime': [''], 'text': [''], 'BodyPart': [f'{body_part}'], 'PatientID': [f'{prev_patient_id}']})    # Append body part and patient info of the next measurement
                try:
                    for col in filtered_df.columns:
                        if col not in exu95_row.columns:
                            exu95_row[col] = pd.NA
                    exu95_row = exu95_row[filtered_df.columns]
                    filtered_df = pd.concat([filtered_df, exu95_row], ignore_index=True)
                except Exception as e:
                    print("Append failed:", e)
                # Reset variables for the next block
                start_block = False 
                
    # Reset index of the filtered dataframe
    filtered_df.reset_index(drop=True, inplace=True)
    return filtered_df, first_ptab


def join_events(df):
    
    # Find indices where sourceid is "mri_frr_264" and "mri_frr_265"
    indices_264 = df.index[df['sourceID'] == 'MRI_FRR_264'].tolist()
    indices_265 = df.index[df['sourceID'] == 'MRI_FRR_265'].tolist()

    # Find pairs of consecutive "mri_frr_264" and "mri_frr_265" rows
    to_delete = []
    count=0
    for idx_264 in indices_264:
        df.at[idx_264, 'text'] = df.at[idx_264, 'text'] +' '+ df.at[idx_264+1, 'text']
        count=count+1

    # Drop the "mri_frr_265" rows
    df.drop(indices_265, inplace=True)

    # Reset index if needed
    df.reset_index(drop=True, inplace=True)

    # Step 2: Check if every 'MRI_FRR_264' is followed by 'MRI_FRR_265'
    invalid_indices = []
    valid_indices=0
    tt_indices=0
    for idx in indices_264:
        tt_indices+=1
        next_idx = idx + 1
        if next_idx not in indices_265:  # If next row is NOT 'MRI_FRR_265'
            invalid_indices.append(idx)
        else:
            valid_indices+=1

    # Step 3: Print results
    if not invalid_indices:
        print(f"All 'MRI_FRR_264' are correctly followed by 'MRI_FRR_265':{valid_indices}")
    else:
        print(f"Found {len(invalid_indices)} 'MRI_FRR_264' NOT followed by 'MRI_FRR_265' at row indices: {invalid_indices}. total mri_frr_264 indices: {tt_indices}")    
    return df


def to_bincolumns(row):
    text = row['text']
    if pd.notna(text):
        # Initialize to NaN if no corresponding values found
        z_in = np.nan
        z_out = np.nan
        y_down = np.nan
        y_up = np.nan

        # Check for specific substrings and update accordingly
        if 'ZAxisInPossible: 1' in text:
            z_in = 1
        elif 'ZAxisInPossible: 0' in text:
            z_in = 0

        if 'ZAxisOutPossible: 1' in text:
            z_out = 1
        elif 'ZAxisOutPossible: 0' in text:
            z_out = 0

        if 'YAxisDownPossible: 1' in text:
            y_down = 1
        elif 'YAxisDownPossible: 0' in text:
            y_down = 0

        if 'YAxisUpPossible: 1' in text:
            y_up = 1
        elif 'YAxisUpPossible: 0' in text:
            y_up = 0

        # Update row with extracted values
        row['ZAxisInPossible'] = z_in
        row['ZAxisOutPossible'] = z_out
        row['YAxisDownPossible'] = y_down
        row['YAxisUpPossible'] = y_up

    return row


def ptab(df, first_ptab):
    start = True
    df['PTAB'] = np.nan
    for idx, row in df.iterrows():
        if start and idx>0:
            df['PTAB'].iloc[idx] = first_ptab
        if row['sourceID'] == "MRI_FRR_257":
            ptab = row['text'].split()[-1]  # position appears on the last position of the string
            df['PTAB'].iloc[idx] = ptab
            start = False
    return df



def coils(df):
    df_copy = df.copy()
    total_coils = []
    for idx, row in df.iterrows():
        if row.sourceID == 'MRI_CCS_11':     # Event containing coil elements
            coils = expand_coils(row.text) # coils is a list in the format: S1, S2, S3, HC1 ...
            for coil in coils:
                if coil not in df_copy.columns:
                    df_copy[coil] = np.nan
                    df_copy.at[idx, coil] = 1
                    total_coils.append(coil)
                else:
                    df_copy.at[idx, coil] = 1
    # Find indices of rows where 'sourceid' is 'mri_ccs_11'
    indices = df_copy.index[df_copy['sourceID'] == 'MRI_CCS_11'].tolist()

    # Iterate over the identified indices and copy values #### !!! EXPLAIN IN GUIDE !!!! ####
    for i in range(len(indices) - 1):
        start_idx = indices[i]
        end_idx = indices[i + 1]
        df_copy.loc[start_idx + 1:end_idx-1, total_coils] = df_copy.loc[start_idx, total_coils].values

    # Handle the last segment if needed (from last 'mri_ccs_11' to the end of the DataFrame)
    if indices:
        last_start_idx = indices[-1]
        df_copy.loc[last_start_idx + 1:, total_coils] = df_copy.loc[last_start_idx, total_coils].values
    df_copy[total_coils] = df_copy[total_coils].fillna(0)
    return df_copy



def expand_coils(coils):

    # Remove the initial part and split by commas
    # Extract the coil elements part of the string
    coil_elements_match = re.search(r'Connected coil elements: ([^.)]*)', coils)
    coil_elements_str = coil_elements_match.group(1) if coil_elements_match else ""

    # Split by commas to get individual elements
    elements = coil_elements_str.split(',')
    # Initialize the result list
    result = []

    # Iterate over the elements
    for element in elements:
        element = element.strip()     # Remove spaces at the end and beginning of the string
        if not element.isnumeric():
            # This is a base element with a letter and number
            prefix = re.match(r'[A-Za-z]+', element)
            if prefix:
                prefix = prefix.group()
                prev_prefix = prefix
                if not '-' in element:
                    result.append(element)
            else:
                if '-' in element:
                    # This is a range, expand it
                    start, end = element.split('-')
                    for i in range(int(start), int(end) + 1):
                        result.append(f"{prev_prefix}{i}")
                else:
                    # coil element is on the form of "18K"
                    result.append(element)
            if prefix and '-' in element:
                numbers = re.findall(r'\d+', element)
                for i in range(int(numbers[0]), int(numbers[1]) + 1):
                    result.append(f"{prefix}{i}")
        else:
            # This is a number, combine it with the previous prefix
            result.append(f"{prefix}{element}")
    return result

In [0]:
df_body = pd.read_excel(r'/dbfs/FileStore/tables/bodyupdated.xlsx')
df_body.drop(columns='MRType',inplace=True)
df_body['BodyGroup'] = df_body['BodyGroup'].str.upper()
df_body['BodyPart'] = df_body['BodyPart'].str.upper()

eventlog_df = spark.read.table("hive_metastore.eventlog.common_eventlog").select(
        F.date_format(F.col("EventDateTime"), "yyyy-MM-dd HH:mm:ss").alias("EventDateTime"),
        "SerialNumber",
        "MessageIdentification",
        "TimeZoneOffset",
        "Message",
        "Line",
        #"SerialNumber"
    )
examination_df = spark.read.table("hive_metastore.examination.examination_workflow")

SERIAL_NUMBER = "202594"

#TARGET_DATE = "2024-04-16" 
START_DATE = "2024-04-16"
END_DATE = "2024-04-16" 
SOURCEID_LIST = [
    "MRI_FRR_2", "MRI_FRR_265", "MRI_FRR_264", 
    "MRI_CCS_11", "MRI_FRR_257", "MRI_MPT_1005", "MRI_FRR_256",
    "MRI_FRR_34", "MRI_FRR_3", "MRI_MSR_100", "MRI_MSR_104",
    "MRI_EXU_95", "MRI_MSR_18", "MRI_FRR_18", "MRI_MSR_21",
    "MRI_MSR_34", "MRI_MSR_26", "MRI_MSR_22", "MRI_MSR_40",
    "MRI_MSR_25", "MRI_MSR_24"
]

result_df = (
    eventlog_df
    .filter(F.col("SerialNumber") == SERIAL_NUMBER)
    # Apply timezone offset
    .withColumn(
        "AdjustedEventDateTime", 
        F.timestamp_seconds(F.unix_timestamp(F.col("EventDateTime")) + F.col("TimeZoneOffset"))
    )
    # Now filter for target date based on adjusted local time
    .filter(F.to_date(F.col("AdjustedEventDateTime")).between(START_DATE, END_DATE))
    .filter(F.col("MessageIdentification").isin(SOURCEID_LIST))
    # Final select
    .select(
        F.date_format(F.col("AdjustedEventDateTime"), "yyyy-MM-dd HH:mm:ss").alias("AdjustedEventDateTime"),
        "MessageIdentification",
        "Message",
        "Line",
        "SerialNumber"
    )
    # Sort by time within the day
    .orderBy(F.col("AdjustedEventDateTime").asc())
)

# Rename columns to match downstream expectations
result_df = result_df.toDF("datetime", "sourceID", "text", "Line", "SerialNumber")

pandas_df = result_df.toPandas()

df_sorted_pandas = (
    pandas_df.groupby("datetime", group_keys=False)
      .apply(lambda g: g.sort_values("Line"))
      .reset_index(drop=True)
)

df_filter, first_ptab = interpatient(df_sorted_pandas)
if not df_filter.empty:
    df_join = join_events(df_filter)
    df_join.drop_duplicates(subset=['sourceID', 'datetime','PatientID'], keep='first', inplace=True)
    df_join.reset_index(drop=True, inplace=True)
    df_bool = df_join.apply(to_bincolumns, axis=1)
    df_ptab = ptab(df_bool, first_ptab)
    columns_to_ffill = ['ZAxisInPossible', 'ZAxisOutPossible', 'YAxisDownPossible', 'YAxisUpPossible', 'PTAB']
    df_ptab[columns_to_ffill] = df_ptab[columns_to_ffill].fillna(method='ffill')
    df_coils = coils(df_ptab)
    #df_coils['SN'] = file.split('.')[0][2:]
    df_coils['SN'] = '202594'
    exchange = pd.DataFrame()
    exchange = pd.concat([exchange, df_coils], axis=0)

if not exchange.empty and 'BodyPart' in exchange.columns:
    result = pd.merge(exchange, df_body, on="BodyPart", how='left')
else:
    print("No valid patient exchanges found")
    result = pd.DataFrame() 

body_parts = result['BodyPart'].notnull()

result['BodyPart_from'] = result['BodyPart'].where(body_parts).ffill()
result['BodyPart_to'] = result['BodyPart'].where(body_parts).bfill()
result['BodyGroup_from'] = result['BodyGroup'].where(body_parts).ffill()
result['BodyGroup_to'] = result['BodyGroup'].where(body_parts).bfill()
result['PatientID_from'] = result['PatientID'].where(body_parts).ffill()
result['PatientID_to'] = result['PatientID'].where(body_parts).bfill()

df_filter_final = result[result['BodyPart'].isnull()].drop(columns=['BodyPart','PatientID', 'BodyGroup'])

examination_df_filtered = (
    examination_df
    .filter(F.col("SerialNumber").contains("202594"))
    .filter(F.col("Year") == '2024')
    #.filter(F.col("Month").isin('4','5','6'))
    .filter(F.col("Month") == '4')
    .filter((F.col("Day") >= '16') & (F.col("Day") <= '17'))
    .select(
        "SerialNumber",
        "WorkflowValues"
    )
)

df_dedup = (
    examination_df_filtered
    .withColumn("PatientId", col("WorkflowValues")["PatientId"])
    .dropDuplicates(["PatientId"])
)

# Define schema for WorkflowValues JSON
workflow_schema = StructType([
    StructField("RegisteredWeight", StringType(), True),
    StructField("Position", StringType(), True),
    StructField("Sequence", StringType(), True),
    StructField("Weight", StringType(), True),
    StructField("MeasUID", StringType(), True),
    StructField("Age", StringType(), True),
    StructField("Height", StringType(), True),
    StructField("ProtocolName", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("PatientId", StringType(), True),
    StructField("Direction", StringType(), True)
])


examination_df_filtered_flat = df_dedup.select(
    col("SerialNumber"),
    col("WorkflowValues")["PatientId"].alias("PatientId"),
    col("WorkflowValues")["Position"].alias("Position"),
    col("WorkflowValues")["Weight"].alias("Weight"),
    col("WorkflowValues")["Age"].alias("Age"),
    col("WorkflowValues")["Height"].alias("Height"),
    col("WorkflowValues")["Direction"].alias("Direction")
)

pandas_examination_df_filtered = examination_df_filtered_flat.toPandas()

df_merged = df_filter_final.merge(
    pandas_examination_df_filtered[["PatientId", "Position", "Weight", "Age", "Height", "Direction"]],
    how="left",
    left_on="PatientID_to",
    right_on="PatientId"
)


In [0]:
csv_path = f'/dbfs/FileStore/pat_exchange/DATA_{SERIAL_NUMBER}.csv'

if not os.path.exists(csv_path):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df_merged.to_csv(csv_path, index=False, header=True)
    print(f"Saved: {csv_path}")
else:
    print(f"File already exists: {csv_path}")

In [0]:
displayHTML(f'<a href="/files/pat_exchange/DATA_{SERIAL_NUMBER}.csv" download>Download DATA_{SERIAL_NUMBER}.csv</a>')